# RAMP — Storybook agent

**Step 1 (terminal, once):** pull + save the data
```
poetry run python scripts/dump_repo_context.py <repo_id>
```
That writes `notebooks/data/repo_context_<connected_repo_id>.json`.

Schema:
- `file_nodes` / `file_edges` — folder/file dependency graph
- `function_nodes` / `function_edges` — function call graph (with `source_code`)
- `overview_stats` — files / folders / functions / dependencies counts

In [1]:
import glob
import json
from pathlib import Path

# --- Load the most recently saved repo_context_*.json ---
NOTEBOOKS = Path("notebooks") if (Path("notebooks") / "data").exists() else Path(".")
files = sorted(
    glob.glob(str(NOTEBOOKS / "data" / "repo_context_*.json")),
    key=lambda p: Path(p).stat().st_mtime,
    reverse=True,
)
if not files:
    raise FileNotFoundError(
        "No repo_context_*.json found. Run first:\n"
        "  poetry run python scripts/dump_repo_context.py <repo_id>"
    )

# Or pin a specific file: DATA_PATH = NOTEBOOKS / "data" / "repo_context_<id>.json"
DATA_PATH = Path(files[0])
print(f"Loading: {DATA_PATH}")
context = json.loads(DATA_PATH.read_text())
context["__data_path__"] = str(DATA_PATH)

Loading: data/repo_context_a9954d1a-6308-46a9-bc20-4042b2be2bf1.json


In [2]:
from typing import Optional
from pydantic import BaseModel, Field


class GraphNode(BaseModel):
    id: str
    label: str
    node_type: str
    file_path: Optional[str] = None
    file_id: Optional[str] = None
    line_start: Optional[int] = None
    line_end: Optional[int] = None
    is_async: bool = False
    source_code: Optional[str] = None
    symbol_type: Optional[str] = None
    return_type: Optional[str] = None
    parameters: Optional[str] = None
    docstring: Optional[str] = None


class GraphEdge(BaseModel):
    source: str
    target: str
    edge_type: str
    call_line: Optional[int] = None
    weight: int = 1


class OverviewStats(BaseModel):
    files: int = 0
    folders: int = 0
    functions: int = 0
    dependencies: int = 0


class RepoContext(BaseModel):
    connected_repo_id: str
    repo_id: Optional[str] = None
    full_name: Optional[str] = None
    default_branch: Optional[str] = None
    language: Optional[str] = None
    description: Optional[str] = None
    overview_stats: OverviewStats = OverviewStats()
    file_nodes: list[GraphNode] = Field(default_factory=list)
    file_edges: list[GraphEdge] = Field(default_factory=list)
    function_nodes: list[GraphNode] = Field(default_factory=list)
    function_edges: list[GraphEdge] = Field(default_factory=list)


# Typed view of the loaded JSON (drops the __data_path__ helper key)
ctx = RepoContext(**{k: v for k, v in context.items() if k != "__data_path__"})

In [3]:
s = ctx.overview_stats
print(f"Repo       : {ctx.full_name} ({ctx.connected_repo_id})")
print(f"Language   : {ctx.language}")
print(f"Branch     : {ctx.default_branch}")
print("---")
print(f"Files      : {s.files}")
print(f"Folders    : {s.folders}")
print(f"Functions  : {s.functions}")
print(f"File deps  : {s.dependencies}")
print(f"Call edges : {len(ctx.function_edges)}")

Repo       : macphailmagwira/agent_app (a9954d1a-6308-46a9-bc20-4042b2be2bf1)
Language   : TypeScript
Branch     : main
---
Files      : 24
Folders    : 8
Functions  : 29
File deps  : 27
Call edges : 17


In [4]:
# --- Inspect a sample function node (with source code) ---
if ctx.function_nodes:
    fn = ctx.function_nodes[0]
    print(f"Function: {fn.label}  ({fn.node_type})  async={fn.is_async}")
    print(f"File     : {fn.file_path}:{fn.line_start}-{fn.line_end}")
    print("Source:")
    print(fn.source_code or "(no source captured)")

Function: renderContent  (function)  async=False
File     : src/App.tsx:39-None
Source:
  const renderContent = () => {
    switch (activeTab) {
      case 'dashboard':
        return <Dashboard onNavigate={setActiveTab} />;
      case 'pos':
        return <POS />;
      case 'orders':
        return <Orders />;
      case 'batches':
        return <Batches />;
      default:
        return <Dashboard onNavigate={setActiveTab} />;
    }
  };

  return (
    <Layout activeTab={activeTab} setActiveTab={setActiveTab}>
      {renderContent()}
    </Layout>
  );
};

const App: React.FC = () => {
  return (
    <AppProvider>
      <AppContent />
    </AppProvider>
  );
};

export default App;


In [5]:
# --- Processing example: prepare inputs for the story/flow agent ---
# Builds the function_nodes (+source) + function_edges payload the AI
# services consume, plus a couple of derived views.

flow_nodes = [
    {
        "id": n.id,
        "label": n.label,
        "node_type": n.node_type,
        "file_path": n.file_path,
        "is_async": n.is_async,
        "line_start": n.line_start,
        "source_code": n.source_code,
    }
    for n in ctx.function_nodes
]
flow_edges = [
    {"source": e.source, "target": e.target, "edge_type": e.edge_type}
    for e in ctx.function_edges
]

async_fns = [n.label for n in ctx.function_nodes if n.is_async]
by_file: dict[str, int] = {}
for n in ctx.function_nodes:
    if n.file_path:
        by_file[n.file_path] = by_file.get(n.file_path, 0) + 1

print(f"Async functions ({len(async_fns)}): {async_fns[:10]}")
print("\nFunctions per file (top 10):")
for path, count in sorted(by_file.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"  {count:3d}  {path}")

print(f"\nPrepared flow_nodes={len(flow_nodes)} flow_edges={len(flow_edges)}")
# `flow_nodes` + `flow_edges` are now ready to pass into the LangGraph agent.

Async functions (13): ['handleCreate', 'handleLogout', 'handleSubmit', 'handleRefresh', 'handleCheckout', 'refreshData', 'login', 'logout', 'createBatch', 'createOrder']

Functions per file (top 10):
   12  src/context/AppContext.tsx
    5  src/components/Layout.tsx
    3  src/hooks/useAgent.ts
    2  src/components/Batches.tsx
    2  src/components/Orders.tsx
    2  src/components/POS.tsx
    1  src/App.tsx
    1  src/components/Login.tsx
    1  src/pages/Dashboard.tsx

Prepared flow_nodes=29 flow_edges=17


## Step 2 — Generate the Storybook

The extracted `context` above is now fed into the **LangGraph Storybook agent** (`src/features/ai/story_book_agent.py`). It runs three subgraphs — **Project Setup**, **Everyday Tasks**, and **Feature Exploration** (which also shows how each feature evolved over time) — and persists the result to `notebooks/data/story_book_<repo_id>.json`.

In [ ]:
import json, urllib.request

# The Storybook agent now lives in the API. Send the extracted context to it.
payload = json.dumps({"context": context, "intent": "onboarding"}).encode()
req = urllib.request.Request(
    "http://localhost:8000/api/v1/ai/story-book",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=900) as resp:
        book = json.loads(resp.read())
    print("Storybook generated:", book.get("full_name"),
          "| flows:", (book.get("features") or {}).get("flow_count"))
    print("Saved to notebooks/data/story_book_%s.json" % book.get("repo_id"))
except Exception as e:
    print("API not running or error:", e)
    print("Run in-process instead:")
    print("  from src.features.ai.story_book_agent import generate_story_book")
    print("  book = await generate_story_book(context)")